# StyleGAN3 Training on Kaggle (Metacreation-Lab fork)

This notebook clones and trains [Metacreation-Lab/stylegan3](https://github.com/Metacreation-Lab/stylegan3) on Kaggle.

**What it does:**
- Clones the repo and installs dependencies
- Reads your training images from a **zip file stored in a Kaggle Dataset**
- Optionally resumes training from a **`.pkl` checkpoint stored in a Kaggle Dataset**
- Runs `train.py`, writing snapshot `.pkl` files as it goes
- Copies snapshot `.pkl` files into a clean output folder so you can save them as a **new/updated Kaggle Dataset** for use in a later run (resume, generate, etc.)

**Before running:**
1. Attach your images dataset (containing a `.zip` of training images) to this notebook via *Add Data*.
2. (Optional) Attach a dataset containing a previous `.pkl` checkpoint if you want to resume training.
3. Edit the `CONFIG` cell below to match your dataset paths and training settings.
4. Turn on a GPU accelerator (Settings -> Accelerator -> GPU T4 x2, or similar).


## 1. Install dependencies & clone the repo

In [1]:
import os, subprocess, sys

REPO_URL = "https://github.com/Metacreation-Lab/stylegan3.git"
REPO_DIR = "/kaggle/working/stylegan3"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo already cloned at", REPO_DIR)

# Core deps for StyleGAN3 (matches the NVlabs/stylegan3 requirements the fork is based on)
pkgs = [
    "ninja", "click", "requests", "tqdm", "pyspng",
    "imageio", "imageio-ffmpeg==0.4.3", "psutil", "scipy",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

print("torch:", __import__("torch").__version__)
print("CUDA available:", __import__("torch").cuda.is_available())


Cloning into '/kaggle/working/stylegan3'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9/26.9 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 15.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


torch: 2.10.0+cu128
CUDA available: True


## 1b. Compatibility patches for the newer PyTorch in this Kaggle image

The cloned repo assumes an older PyTorch API in a couple of places. The cell below patches both automatically after every fresh clone:

1. **`torch_utils/misc.py`** - `InfiniteSampler` calls `super().__init__(dataset)`, but `torch.utils.data.Sampler.__init__` no longer accepts a `dataset` argument. This throws:
   ```
   TypeError: object.__init__() takes exactly one argument (the instance to initialize)
   ```

2. **`train.py`** - the optimizer betas are hardcoded as `betas=[0,0.99]` (an `int` mixed with a `float`). Newer PyTorch's `Adam` requires both to be floats, otherwise it throws:
   ```
   ValueError: betas must be either both floats or both Tensors
   ```


In [2]:
misc_path = os.path.join(REPO_DIR, "torch_utils", "misc.py")

with open(misc_path, "r") as f:
    misc_src = f.read()

old_line = "super().__init__(dataset)"
new_line = "super().__init__()"

if old_line in misc_src:
    misc_src = misc_src.replace(old_line, new_line)
    with open(misc_path, "w") as f:
        f.write(misc_src)
    print(f"Patched {misc_path}: replaced '{old_line}' with '{new_line}'")
elif new_line in misc_src:
    print(f"{misc_path} already patched - nothing to do.")
else:
    print(f"WARNING: expected line not found in {misc_path} - check the file manually.")

# Newer PyTorch requires Adam's betas to be both floats (not int/float mixed).
# The repo hardcodes betas=[0, 0.99], which now raises:
#   ValueError: betas must be either both floats or both Tensors
train_py_path = os.path.join(REPO_DIR, "train.py")

with open(train_py_path, "r") as f:
    train_src = f.read()

old_betas = "betas=[0,0.99]"
new_betas = "betas=[0.0,0.99]"

count = train_src.count(old_betas)
if count:
    train_src = train_src.replace(old_betas, new_betas)
    with open(train_py_path, "w") as f:
        f.write(train_src)
    print(f"Patched {train_py_path}: replaced {count} occurrence(s) of '{old_betas}' with '{new_betas}'")
elif new_betas in train_src:
    print(f"{train_py_path} already patched - nothing to do.")
else:
    print(f"WARNING: expected betas line not found in {train_py_path} - check the file manually.")


Patched /kaggle/working/stylegan3/torch_utils/misc.py: replaced 'super().__init__(dataset)' with 'super().__init__()'
Patched /kaggle/working/stylegan3/train.py: replaced 2 occurrence(s) of 'betas=[0,0.99]' with 'betas=[0.0,0.99]'


## 2. Configuration

Edit the paths below.

- `IMAGES_SOURCE_PATH`: full path to your training images. Can be either a **folder of images** (e.g. an
  already-unzipped Kaggle Dataset folder) or a **`.zip` file** - `dataset_tool.py` accepts both. Since you'll
  have multiple image datasets over time, just point this at whichever one you want for this run.
- `RESUME_DATASET_DIR`: the folder under `/kaggle/input/` for the Kaggle Dataset that holds a previous **`.pkl` checkpoint** (set to `None` to train from scratch).
- `RESUME_PKL_NAME`: set to `None` to auto-pick the newest `.pkl` in that dataset, or give an exact filename.
- `PKL_OUTPUT_DIR`: where snapshot `.pkl` files get collected in `/kaggle/working` so they persist in the notebook's output (which you can then save as a new Kaggle Dataset version).


In [3]:
# ---- EDIT THESE ----

# Full path to a folder of training images OR a .zip of them (dataset_tool.py accepts either).
# Point this at whichever attached dataset/folder you want to train on for this run.
IMAGES_SOURCE_PATH = "/kaggle/input/datasets/michel64leblond/autolumetrainingimages/SunsetsWB/SunsetsWB"

# Dataset containing a previous .pkl checkpoint to resume from (or None to train from scratch)
RESUME_DATASET_DIR = None  #"/kaggle/input/piklefiles"  # set to None to disable
RESUME_PKL_NAME = None #"network-snapshot-000256.pkl"  # e.g. "network-snapshot-002400.pkl" or None to auto-pick the newest
RESUME_PKL_PATH = None #"/kaggle/input/datasets/michel64leblond/generatedpkls/"#None

# Where prepared training data + snapshots get written during this run
WORK_DIR = "/kaggle/working"
DATASET_ZIP_OUT = f"{WORK_DIR}/dataset.zip"          # dataset_tool.py output, used to train
RUNS_DIR = f"{WORK_DIR}/training-runs"                # raw stylegan3 training-run output
PKL_OUTPUT_DIR = f"{WORK_DIR}/pkl-output"             # flat folder of .pkl snapshots to persist as a Dataset

# Training hyperparameters
RESOLUTION = 512          # must match your images - your training_dataset_256x256 folder implies 256
CFG = "stylegan3-t"       # "stylegan3-t" (translation-equiv) or "stylegan3-r" (rotation-equiv) or "stylegan2"
GPUS = 2                  # match the accelerator you selected (e.g. 2 for T4 x2)
BATCH_SIZE = 16           # total batch size across all GPUs
KIMG = 3000               # total training length in thousands of images (reduce for short Kaggle sessions)
SNAPSHOT_TICKS = 4        # save a .pkl every N ticks (lower = more frequent, more disk usage)
MIRROR = True             # horizontal flip augmentation
COMPUTE_METRICS = False   # True = compute fid50k_full at every snapshot (adds ~10-30+ min pause each time,
                          # independent of dataset size). False = skip metrics entirely - training runs
                          # straight through; .pkl snapshots and fakes*.png sample grids are unaffected.

os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(PKL_OUTPUT_DIR, exist_ok=True)

if not os.path.exists(IMAGES_SOURCE_PATH):
    raise FileNotFoundError(f"IMAGES_SOURCE_PATH does not exist: {IMAGES_SOURCE_PATH}")
print("Training images source:", IMAGES_SOURCE_PATH)
print("Config set.")


Training images source: /kaggle/input/datasets/michel64leblond/autolumetrainingimages/SunsetsWB/SunsetsWB
Config set.


## 3. Prepare the dataset with `dataset_tool.py`

`dataset_tool.py` reads directly from `IMAGES_SOURCE_PATH` (folder or zip - no manual unzip step needed) and
writes a training-ready zip to `DATASET_ZIP_OUT`. Since your images are already `256x256`, matching
`RESOLUTION` above means `dataset_tool.py` just repackages them (no resizing) - it's still a required step
because `train.py` needs the accompanying `dataset.json` labels file that this generates.


In [4]:
def find_resume_pkl(dataset_dir, explicit_name):
    if dataset_dir is None:
        return None
    if not os.path.isdir(dataset_dir):
        print(f"Resume dataset dir {dataset_dir} not found - training from scratch.")
        return None
    if explicit_name:
        candidate = os.path.join(dataset_dir, explicit_name)
        if not os.path.exists(candidate):
            raise FileNotFoundError(f"Could not find {candidate}")
        return candidate
    pkls = glob.glob(os.path.join(dataset_dir, "**", "*.pkl"), recursive=True)
    if not pkls:
        print(f"No .pkl files found under {dataset_dir} - training from scratch.")
        return None
    # newest by filename (network-snapshot-XXXXXX.pkl sorts naturally) then by mtime as a fallback
    pkls.sort(key=lambda p: (p, os.path.getmtime(p)))
    return pkls[-1]

RESUME_PKL_PATH = find_resume_pkl(RESUME_DATASET_DIR, RESUME_PKL_NAME)
print("Resuming from:", RESUME_PKL_PATH if RESUME_PKL_PATH else "(training from scratch)")


Resuming from: (training from scratch)


In [5]:
import glob

dataset_tool = os.path.join(REPO_DIR, "dataset_tool.py")

cmd = [
    sys.executable, dataset_tool,
    f"--source={IMAGES_SOURCE_PATH}",
    f"--dest={DATASET_ZIP_OUT}",
    f"--resolution={RESOLUTION}x{RESOLUTION}",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Prepared dataset at:", DATASET_ZIP_OUT)


Running: /usr/bin/python3 /kaggle/working/stylegan3/dataset_tool.py --source=/kaggle/input/datasets/michel64leblond/autolumetrainingimages/SunsetsWB/SunsetsWB --dest=/kaggle/working/dataset.zip --resolution=512x512


  0%|          | 0/128 [00:00<?, ?it/s]/kaggle/working/stylegan3/dataset_tool.py:444: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PIL.Image.fromarray(img, { 1: 'L', 3: 'RGB' }[channels])
 99%|█████████▉| 127/128 [00:49<00:00,  2.69it/s]

Prepared dataset at: /kaggle/working/dataset.zip


100%|██████████| 128/128 [00:49<00:00,  2.59it/s]


## 4. Locate a `.pkl` checkpoint to resume from (optional)

In [ ]:
train_py = os.path.join(REPO_DIR, "train.py")

cmd = [
    sys.executable, train_py,
    f"--outdir={RUNS_DIR}",
    f"--cfg={CFG}",
    f"--data={DATASET_ZIP_OUT}",
    f"--gpus={GPUS}",
    f"--batch={BATCH_SIZE}",
    f"--gamma=8.2",
    f"--kimg={KIMG}",
    f"--snap={SNAPSHOT_TICKS}",
    f"--mirror={1 if MIRROR else 0}",
    f"--metrics={'fid50k_full' if COMPUTE_METRICS else 'none'}",
]
if RESUME_PKL_PATH:
    cmd.append(f"--resume={RESUME_PKL_PATH}")

print("Running:", " ".join(cmd))

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()
print("Training process exited with code", process.returncode)


Running: /usr/bin/python3 /kaggle/working/stylegan3/train.py --outdir=/kaggle/working/training-runs --cfg=stylegan3-t --data=/kaggle/working/dataset.zip --gpus=2 --batch=16 --gamma=8.2 --kimg=3000 --snap=4 --mirror=1 --metrics=none

Training options:
{
  "G_kwargs": {
    "class_name": "training.networks_stylegan3.Generator",
    "z_dim": 512,
    "w_dim": 512,
    "mapping_kwargs": {
      "num_layers": 2
    },
    "channel_base": 32768,
    "channel_max": 512,
    "magnitude_ema_beta": 0.9994456359721023
  },
  "D_kwargs": {
    "class_name": "training.networks_stylegan2.Discriminator",
    "block_kwargs": {
      "freeze_layers": 0
    },
    "mapping_kwargs": {},
    "epilogue_kwargs": {
      "mbstd_group_size": 4
    },
    "channel_base": 32768,
    "channel_max": 512
  },
  "G_opt_kwargs": {
    "class_name": "torch.optim.Adam",
    "betas": [
      0.0,
      0.99
    ],
    "eps": 1e-08,
    "lr": 0.0025
  },
  "D_opt_kwargs": {
    "class_name": "torch.optim.Adam",
    "bet

## 5. Train

Runs `train.py` from the repo, streaming output live. Snapshot `.pkl` files land under `RUNS_DIR/<run-name>/`.


## 6. Collect snapshot `.pkl` files into a persistent output folder

Copies every `.pkl` produced during training into `PKL_OUTPUT_DIR` (a flat folder under `/kaggle/working`).
Run this cell any time during/after training to sync the latest snapshots - it's cheap to re-run.


In [ ]:
import shutil

snapshot_pkls = glob.glob(os.path.join(RUNS_DIR, "**", "network-snapshot-*.pkl"), recursive=True)
snapshot_pkls.sort()

copied = []
for src in snapshot_pkls:
    dst = os.path.join(PKL_OUTPUT_DIR, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        copied.append(dst)

print(f"{len(snapshot_pkls)} snapshot(s) found, {len(copied)} newly copied to {PKL_OUTPUT_DIR}")
for p in sorted(os.listdir(PKL_OUTPUT_DIR)):
    print(" -", p)


## 7. Persist `.pkl` files as a Kaggle Dataset for future runs

Two ways to make the checkpoints in `PKL_OUTPUT_DIR` available to a later notebook run:

**Option A - manual (simplest):** click **Save Version** on this notebook. Everything under `/kaggle/working`
(including `PKL_OUTPUT_DIR`) becomes downloadable notebook output, and you can create/update a Kaggle Dataset
from it via *File -> New Dataset* (or *Update* an existing dataset) pointing at `PKL_OUTPUT_DIR`.

**Option B - automatic via the Kaggle API (optional, run the cell below).** This pushes `PKL_OUTPUT_DIR` as a
new version of an existing Dataset without needing to end the session first - useful if you want checkpoints
available mid-run in case the session times out. Requires your Kaggle API token to be available as a Kaggle
Secret named `KAGGLE_KEY` (and username as `KAGGLE_USERNAME`), added via *Add-ons -> Secrets*.


In [ ]:
# Optional: push PKL_OUTPUT_DIR to an existing Kaggle Dataset via the API.
# Leave PUSH_TO_KAGGLE_DATASET = False to skip this.

PUSH_TO_KAGGLE_DATASET = False
TARGET_DATASET_SLUG = "michel64leblond/piklefiles"  # "<your-username>/<dataset-slug>"

if PUSH_TO_KAGGLE_DATASET:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")

    import json as _json
    meta_path = os.path.join(PKL_OUTPUT_DIR, "dataset-metadata.json")
    with open(meta_path, "w") as f:
        _json.dump({
            "title": TARGET_DATASET_SLUG.split("/")[-1],
            "id": TARGET_DATASET_SLUG,
            "licenses": [{"name": "CC0-1.0"}],
        }, f)

    subprocess.run([
        "kaggle", "datasets", "version",
        "-p", PKL_OUTPUT_DIR,
        "-m", "Updated checkpoints from training run",
        "-r", "zip",
    ], check=True)
    print(f"Pushed new version of dataset {TARGET_DATASET_SLUG}")
else:
    print("Skipped - set PUSH_TO_KAGGLE_DATASET = True to enable the API push.")
